In [3]:
import pandas as pd
import numpy as np

# ======================
# 1. LOAD DATA
# ======================
df = pd.read_csv("dataset_p1/9meningitisWithMisingValues/mening_missing_12.csv")

print("Shape:", df.shape)
print(df.head())

# ======================
# 2. ANALIZAR MISSING
# ======================
missing = df.isnull().mean().sort_values(ascending=False)

print("\nMissing ratio:\n", missing)

# ======================
# 3. ELIMINAR COLUMNAS EXTREMAS
# ======================
# eliminar columnas con más del 60% de NaN
df = df.loc[:, df.isnull().mean() < 0.6]

# ======================
# 4. IDENTIFICAR TIPOS
# ======================
df = df.convert_dtypes()

num_cols = df.select_dtypes(include=np.number).columns
cat_cols = df.select_dtypes(include=["object", "string"]).columns

# ======================
# 5. FEATURE ENGINEERING CLAVE
# ======================
# crear flags de missing (MUY IMPORTANTE en medicina)

for col in num_cols:
    df[f"{col}_missing"] = df[col].isnull().astype(int)

# ======================
# 6. IMPUTACIÓN INTELIGENTE
# ======================

# Convertir numéricas a float para evitar errores de tipo con medianas
df[num_cols] = df[num_cols].astype(float)

# Numéricas → mediana
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# Categóricas → "Missing"
df[cat_cols] = df[cat_cols].fillna("Missing")

# ======================
# 7. TARGET
# ======================
# Ajusta según dataset real (ejemplo)
target_col = "Diagnosis"

df["target"] = df[target_col]
df = df.drop(columns=[target_col])

# ======================
# 8. ENCODING
# ======================
df_features = df.drop(columns=["target"])
df_features = pd.get_dummies(df_features, drop_first=True)

# ======================
# 9. FEATURES / TARGET
# ======================
X = df_features
y = df["target"]

print("\nFinal shape:", X.shape)
print("Target distribution:\n", y.value_counts())

Shape: (1200, 14)
   Patient_ID    Age  Gender  WBC_Count  Protein_Level  Glucose_Level  \
0           1  101.0  Female     8624.0           16.0           83.0   
1           2   78.0    Male    22623.0          200.0           41.0   
2           3    8.0  Female    12908.0           39.0            3.0   
3           4  104.0  Female    15072.0           58.0           36.0   
4           5   38.0  Female    18623.0          152.0           34.0   

  Pathogen_Present  Diagnosis    Outcome  Hemoglobin  WBC_Blood_Count  \
0               No      Viral  Recovered        15.0           7269.0   
1               No    Unknown  Recovered        18.0           6532.0   
2               No    Unknown  Recovered        16.0           7417.0   
3              Yes  Bacterial  Recovered         7.0          13792.0   
4              Yes  Bacterial  Recovered         5.0          17054.0   

   Platelets  CRP_Level     Risk_Level  
0   160949.0       71.0  Moderate Risk  
1   371741.0       41.